# TMDB Emotion Intensity Pipeline

Downloads the TMDb movies dataset (~930K movies) via `kagglehub`, extracts movie keywords, and scores them using the NRC Emotion Intensity Lexicon to produce per-movie emotion intensity features (anger, fear, joy, sadness, disgust, surprise, anticipation, trust).

Ensure `.env` has `KAGGLE_API_TOKEN` (or `~/.kaggle` credentials).

In [5]:
import os
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv

# Load .env so KAGGLE_API_TOKEN (e.g. KGAT_...) is available in the notebook
load_dotenv()

# Optional: fail fast with a clear message if Kaggle credentials are missing
def _has_kaggle_credentials() -> bool:
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return True
    kaggle_dir = Path.home() / ".kaggle"
    return any((kaggle_dir / name).exists() for name in ("kaggle.json", "access_token", "access_token.txt"))

if not _has_kaggle_credentials():
    raise RuntimeError(
        "Kaggle credentials not found. Set KAGGLE_API_TOKEN in .env (or KAGGLE_USERNAME + KAGGLE_KEY), "
        "or add kaggle.json to ~/.kaggle. See readme.md."
    )

DATASET = "asaniczka/tmdb-movies-dataset-2023-930k-movies"
NROWS = 10_000  # set to None to load full dataset

# Download full dataset (no file path) — single-file path causes 404 for this dataset
try:
    base_path = kagglehub.dataset_download(DATASET)
except Exception as e:
    err = str(e).lower()
    if "404" in err or "not found" in err:
        raise RuntimeError(
            f"Dataset not found (404). Check that it exists: {DATASET}"
        ) from e
    if "401" in err or "403" in err or "unauthorized" in err or "forbidden" in err:
        raise RuntimeError(
            "Kaggle returned 401/403. Check credentials and accept the dataset terms on the dataset page."
        ) from e
    raise

# Resolve path: kagglehub returns a dir (extracted) or a zip file
pandas_kwargs = {"nrows": NROWS} if NROWS else {}
base = Path(base_path)
if base.is_file() and base.suffix.lower() == ".zip":
    with zipfile.ZipFile(base, "r") as z:
        csv_names = sorted(n for n in z.namelist() if n.lower().endswith(".csv"))
        if not csv_names:
            raise FileNotFoundError(f"No .csv in {base}")
        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f, **pandas_kwargs)
else:
    zips = list(base.rglob("*.zip"))
    if zips:
        with zipfile.ZipFile(sorted(zips)[0], "r") as z:
            csv_names = sorted(n for n in z.namelist() if n.lower().endswith(".csv"))
            if not csv_names:
                raise FileNotFoundError(f"No .csv in {zips[0]}")
            with z.open(csv_names[0]) as f:
                df = pd.read_csv(f, **pandas_kwargs)
    else:
        csvs = list(base.rglob("*.csv"))
        if not csvs:
            raise FileNotFoundError(f"No .zip or .csv under {base_path}")
        df = pd.read_csv(sorted(csvs)[0], **pandas_kwargs)
print(f"Loaded {len(df)} rows × {len(df.columns)} columns.")

Loaded 10000 rows × 24 columns.


In [6]:
df.head(10)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."
5,293660,Deadpool,7.606,28894,Released,2016-02-09,783100000,108,False,/en971MEXui9diirXlogOrPKmsEn.jpg,...,Deadpool,The origin story of former Special Forces oper...,72.735,/zq8Cl3PNIDGU3iWNRoc5nEZ6pCe.jpg,Witness the beginning of a happy ending.,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre ...",United States of America,English,"superhero, anti hero, mercenary, based on comi..."
6,299536,Avengers: Infinity War,8.255,27713,Released,2018-04-25,2052415039,149,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,...,Avengers: Infinity War,As the Avengers and their allies have continue...,154.340,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,An entire universe. Once and for all.,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, s..."
7,550,Fight Club,8.438,27238,Released,1999-10-15,100853753,139,False,/hZkgoQYus5vegHoetLkCJzb17zJ.jpg,...,Fight Club,A ticking-time-bomb insomniac and a slippery s...,69.498,/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg,Mischief. Mayhem. Soap.,Drama,"Regency Enterprises, Fox 2000 Pictures, Taurus...",United States of America,English,"dual identity, rage and hate, based on novel o..."
8,118340,Guardians of the Galaxy,7.906,26638,Released,2014-07-30,772776600,121,False,/uLtVbjvS1O7gXL8lUOwsFOH4man.jpg,...,Guardians of the Galaxy,"Light years from Earth, 26 years after being a...",33.255,/r7vmZjiyZw9rpJMQJdXpjgiCOk9.jpg,All heroes start somewhere.,"Action, Science Fiction, Adventure",Marvel Studios,United States of America,English,"spacec

In [7]:
# All unique TMDB keywords
keywords_series = df["keywords"].dropna().astype(str)
unique_keywords = sorted(
    {kw.strip() for s in keywords_series for kw in s.split(",") if kw.strip()}
)
print(f"Unique TMDB keywords in sample: {len(unique_keywords):,}")
print(unique_keywords[:50])

Unique TMDB keywords in sample: 16,298
['10th century', '11th century', '12th century', '13th century', '14th century', '15th century', '1600s', '1620s', '16th century', '1750s', '1790s', '17th century', '1800s', '1830s', '1840s', '1850s', '1860s', '1870s', '1890s', '18th century', '1900s', '1910s', '1920s', '1930s', '1940s', '1950s', '1960s', '1966-98)', '1970s', '1980s', '1990s', '19th century', '1st century', '1st century bc', '2000s', '2010s', '2030s', '2040s', '2050s', '2060s', '2070s', '2080s', '2090s', '20th century', '22nd century', '23rd century', '24th century', '2nd century', '3d', '3d animation']


## NRC Word-Emotion Association Lexicon

[NRC Emotion Lexicon](https://www.saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm) v0.92: words × associations with eight emotions (anger, fear, anticipation, trust, surprise, sadness, joy, disgust) and two sentiments (negative, positive). File: `data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt` (tab-separated: word, emotion, 0|1). Refresh from source: [NRC-Emotion-Lexicon.zip](https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Lexicon.zip).

In [8]:
import urllib.request
import zipfile
from pathlib import Path

NRC_LEXICON_ZIP = "https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Lexicon.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "NRC-Emotion-Lexicon.zip"
EXTRACT_DIR = DATA_DIR / "NRC-Emotion-Lexicon"
WORDLEVEL_FILE = EXTRACT_DIR / "NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"

if not WORDLEVEL_FILE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(
        NRC_LEXICON_ZIP,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; Python)",
            "Accept": "*/*",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp, open(ZIP_PATH, "wb") as out:
        out.write(resp.read())
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)


In [9]:
from pathlib import Path

import pandas as pd

WORDLEVEL_FILE = Path("data") / "NRC-Emotion-Lexicon" / "NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
# Data starts after 46 lines of header/comments
nrc_long = pd.read_csv(WORDLEVEL_FILE, sep="\t", header=None, names=["word", "emotion", "value"], skiprows=46)
print(f"NRC lexicon: {len(nrc_long):,} rows (word × emotion).")
print(nrc_long.head(15))

NRC lexicon: 141,494 rows (word × emotion).
           word       emotion  value
0   abandonment      positive      0
1   abandonment       sadness      1
2   abandonment      surprise      1
3   abandonment         trust      0
4         abate         anger      0
5         abate  anticipation      0
6         abate       disgust      0
7         abate          fear      0
8         abate           joy      0
9         abate      negative      0
10        abate      positive      0
11        abate       sadness      0
12        abate      surprise      0
13        abate         trust      0
14    abatement         anger      0


In [10]:
# Pivot: one row per word, columns = emotions/sentiments (0/1)
nrc_wide = nrc_long.pivot(index="word", columns="emotion", values="value").reset_index()
print(f"Unique words: {len(nrc_wide):,}. Emotions/sentiments: {list(nrc_wide.columns[1:])}")
nrc_wide.head(10)

Unique words: 14,150. Emotions/sentiments: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'negative', 'positive', 'sadness', 'surprise', 'trust']


emotion,word,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust
0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,abandonment,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,0.0
2,abate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,abatement,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,abba,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,abbot,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6,abbreviate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,abbreviation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,abdomen,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,abdominal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## NRC Emotion Intensity Lexicon

[NRC Emotion Intensity Lexicon](https://www.saifmohammad.com/WebPages/AffectIntensity.htm) v1: real-valued scores (0–1) for eight emotions. [Download zip](https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Intensity-Lexicon.zip); we load the English file `NRC-Emotion-Intensity-Lexicon-v1.txt` (word, emotion, intensity).

In [11]:
import urllib.request
import zipfile
from pathlib import Path

NRC_INTENSITY_ZIP = "https://www.saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Intensity-Lexicon.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "NRC-Emotion-Intensity-Lexicon.zip"
EXTRACT_DIR = DATA_DIR / "NRC-Emotion-Intensity-Lexicon"
INTENSITY_FILE = EXTRACT_DIR / "NRC-Emotion-Intensity-Lexicon-v1.txt"

if not INTENSITY_FILE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(
        NRC_INTENSITY_ZIP,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; Python)",
            "Accept": "*/*",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp, open(ZIP_PATH, "wb") as out:
        out.write(resp.read())
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)


In [12]:
from pathlib import Path

import pandas as pd

INTENSITY_FILE = Path("data") / "NRC-Emotion-Intensity-Lexicon" / "NRC-Emotion-Intensity-Lexicon-v1.txt"
nrc_intensity_long = pd.read_csv(
    INTENSITY_FILE, sep="\t", header=None, names=["word", "emotion", "intensity"]
)
print(f"NRC intensity: {len(nrc_intensity_long):,} rows. Emotions: {nrc_intensity_long['emotion'].unique().tolist()}")
nrc_intensity_long.head(10)

NRC intensity: 9,829 rows. Emotions: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']


,word,emotion,intensity
0,outraged,anger,0.964
1,brutality,anger,0.959
2,hatred,anger,0.953
3,hateful,anger,0.940
4,terrorize,anger,0.939
5,violently,anger,0.938
6,infuriated,anger,0.938
7,furious,anger,0.929
8,furiously,anger,0.927
9,enraged,anger,0.927


In [13]:
# Pivot: one row per word, columns = emotion intensity scores (0–1)
# NaN → 0: lexicon only records non-zero entries; missing = no signal
nrc_intensity_wide = (
    nrc_intensity_long
    .pivot(index="word", columns="emotion", values="intensity")
    .fillna(0)
    .reset_index()
)
print(f"Words with intensity scores: {len(nrc_intensity_wide):,}")
nrc_intensity_wide.head(10)

Words with intensity scores: 5,891


emotion,word,anger,anticipation,disgust,fear,joy,sadness,surprise,trust
0,aaaaaaah,0.000,0.000,0.000,0.344,0.0,0.000,0.000,0.000
1,aaaah,0.000,0.000,0.000,0.234,0.0,0.000,0.000,0.000
2,abacus,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.406
3,abandon,0.000,0.000,0.000,0.531,0.0,0.703,0.000,0.000
4,abandoned,0.222,0.000,0.000,0.534,0.0,0.828,0.000,0.000
5,abandonment,0.438,0.000,0.000,0.609,0.0,0.859,0.523,0.000
6,abbot,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.438
7,abduction,0.000,0.000,0.000,0.700,0.0,0.750,0.773,0.000
8,aberration,0.000,0.000,0.531,0.000,0.0,0.000,0.000,0.000
9,abeyance,0.000,0.391,0.000,0.000,0.0,0.000,0.000,0.000


## Joined Lexicon: binary presence + continuous intensity + sentiment polarity

NRC EmoLex (binary 0/1 per emotion + positive/negative sentiment) and NRC Emotion Intensity Lexicon (real-valued 0–1) share the same word-level ontology. Joining on `word` gives each term: **which emotions it activates**, **how strongly**, and **its overall polarity**.

In [14]:
EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

# Rename intensity columns to avoid collision with binary columns
intensity_renamed = nrc_intensity_wide.rename(
    columns={e: f"{e}_intensity" for e in EMOTIONS}
)

# Outer join: keep words from either lexicon
nrc_joined = nrc_wide.merge(intensity_renamed, on="word", how="outer").fillna(0)

# Cast binary columns back to int
binary_cols = EMOTIONS + ["positive", "negative"]
nrc_joined[binary_cols] = nrc_joined[binary_cols].astype(int)

print(f"Joined lexicon: {len(nrc_joined):,} words")
print(f"Columns: {list(nrc_joined.columns)}")
nrc_joined.head(10)

Joined lexicon: 15,255 words
Columns: ['word', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'negative', 'positive', 'sadness', 'surprise', 'trust', 'anger_intensity', 'anticipation_intensity', 'disgust_intensity', 'fear_intensity', 'joy_intensity', 'sadness_intensity', 'surprise_intensity', 'trust_intensity']


emotion,word,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust,anger_intensity,anticipation_intensity,disgust_intensity,fear_intensity,joy_intensity,sadness_intensity,surprise_intensity,trust_intensity
0,aaaaaaah,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.344,0.0,0.000,0.000,0.000
1,aaaah,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.234,0.0,0.000,0.000,0.000
2,abacus,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.406
3,abandon,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.531,0.0,0.703,0.000,0.000
4,abandoned,0,0,0,0,0,0,0,0,0,0,0.222,0.0,0.0,0.534,0.0,0.828,0.000,0.000
5,abandonment,0,0,0,0,0,0,0,1,1,0,0.438,0.0,0.0,0.609,0.0,0.859,0.523,0.000
6,abate,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.000
7,abatement,0,0,0,0,0,0,0,0,0,0,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.000
8,abba,0,0,0,0,0,0,1,0,0,0,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.000
9,abbot,0,0,0,0,0,0,0,0,0,1,0.000,0.0,0.0,0.000,0.0,0.000,0.000,0.438


In [15]:
# Emotion–sentiment polarity mapping (derived from NRC EmoLex, computed empirically).
# For each emotion, % of words with that emotion also labelled positive/negative:
#
#   emotion        pos%    neg%    signal
#   ─────────────────────────────────────────────────────
#   joy           98.4%    5.4%    strongly positive
#   trust         69.3%    5.8%    positive
#   anticipation  57.5%   21.1%    positive
#   surprise      41.4%   40.8%    mixed
#   anger          4.7%   92.0%    strongly negative
#   disgust        2.8%   91.8%    strongly negative
#   sadness        4.8%   89.7%    strongly negative
#   fear           6.8%   83.4%    negative
#   ─────────────────────────────────────────────────────
#
# Polarity-weighted intensity:
#   {emotion}_positive_intensity = intensity * positive_flag
#   {emotion}_negative_intensity = intensity * negative_flag
#
# Summary scores:
#   mean_positive_intensity = mean({joy,trust,anticipation,surprise}_positive_intensity)
#   mean_negative_intensity = mean({anger,disgust,sadness,fear}_negative_intensity)

EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"]

nrc_scored = nrc_joined.copy()

pos_cols, neg_cols = [], []
for e in EMOTIONS:
    pos_col = f"{e}_positive_intensity"
    neg_col = f"{e}_negative_intensity"
    nrc_scored[pos_col] = nrc_scored[f"{e}_intensity"] * nrc_scored["positive"]
    nrc_scored[neg_col] = nrc_scored[f"{e}_intensity"] * nrc_scored["negative"]
    pos_cols.append(pos_col)
    neg_cols.append(neg_col)

nrc_scored["mean_positive_intensity"] = nrc_scored[pos_cols].mean(axis=1)
nrc_scored["mean_negative_intensity"] = nrc_scored[neg_cols].mean(axis=1)

display_cols = ["word"] + pos_cols + neg_cols + ["mean_positive_intensity", "mean_negative_intensity"]
print(f"Columns added: {len(pos_cols + neg_cols) + 2}")

examples = ["rage", "celebrate", "betrayal", "wedding", "murder", "hope", "surprise"]
mask = nrc_scored["word"].isin(examples)
nrc_scored[mask][["word"] + [f"{e}_intensity" for e in EMOTIONS] + ["mean_positive_intensity", "mean_negative_intensity"]].set_index("word")

Columns added: 18


emotion,anger_intensity,anticipation_intensity,disgust_intensity,fear_intensity,joy_intensity,sadness_intensity,surprise_intensity,trust_intensity,mean_positive_intensity,mean_negative_intensity
word,,,,,,,,,,
betrayal,0.561,0.000,0.617,0.000,0.000,0.734,0.000,0.00,0.00000,0.239000
celebrate,0.000,0.000,0.000,0.000,0.844,0.000,0.000,0.00,0.00000,0.000000
hope,0.000,0.773,0.000,0.000,0.586,0.000,0.367,0.68,0.30075,0.000000
murder,0.897,0.000,0.839,0.906,0.000,0.828,0.617,0.00,0.00000,0.510875
rage,0.911,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.00000,0.113875
surprise,0.000,0.000,0.000,0.172,0.562,0.000,0.930,0.00,0.20800,0.000000
wedding,0.000,0.000,0.000,0.000,0.641,0.000,0.000,0.00,0.00000,0.000000


In [16]:
# Keyword coverage against the joined lexicon
nrc_words = set(nrc_joined["word"])
direct_matches = [kw for kw in unique_keywords if kw in nrc_words]
token_matches = [
    kw for kw in unique_keywords
    if any(tok in nrc_words for tok in kw.split())
]
print(f"Total unique TMDB keywords:     {len(unique_keywords):>6,}")
print(f"Direct matches (joined lexicon):{len(direct_matches):>6,} ({100*len(direct_matches)/len(unique_keywords):.1f}%)")
print(f"At least one token matches:     {len(token_matches):>6,} ({100*len(token_matches)/len(unique_keywords):.1f}%)")
print(f"\nSample unmatched:")
print([kw for kw in unique_keywords if kw not in nrc_words][:20])

Total unique TMDB keywords:     16,298
Direct matches (joined lexicon): 3,260 (20.0%)
At least one token matches:     11,753 (72.1%)

Sample unmatched:
['10th century', '11th century', '12th century', '13th century', '14th century', '15th century', '1600s', '1620s', '16th century', '1750s', '1790s', '17th century', '1800s', '1830s', '1840s', '1850s', '1860s', '1870s', '1890s', '18th century']


## Phrase-Level Keyword Scoring

TMDB keywords are phrases ("coming of age", "post apocalypse"). The NRC lexicon is word-level, so direct lookup misses most keywords. Fix: tokenize each keyword into words, lemmatize tokens, look up each token in the intensity lexicon, average non-zero token scores per keyword, fall back to zero if no tokens match.

In [17]:
import re
import nltk
from nltk.stem import WordNetLemmatizer

# Download required NLTK data (no-op if already present)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

_lemmatizer = WordNetLemmatizer()


def _tokenize(keyword: str) -> list:
    """Split phrase on whitespace/hyphens/underscores, lowercase, alpha only."""
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]


def _lemma_candidates(tok: str) -> list:
    """Return lemma forms for noun, verb, and adjective POS (deduplicated)."""
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out


def score_keyword_phrase(
    keyword: str,
    intensity_lookup: dict,
    emotions: list,
) -> dict:
    """
    Score a (possibly multi-word) keyword against the NRC intensity lexicon.

    Strategy:
      1. Tokenize keyword into words.
      2. For each token, try lemma forms (noun, verb, adj) in order.
      3. Use the first lemma form found in the intensity lexicon.
      4. Average scores across matched tokens.
      5. Fall back to zero vector if no tokens match.
    """
    tokens = _tokenize(keyword)
    matched_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in intensity_lookup:
                matched_rows.append(intensity_lookup[lemma])
                break
    if not matched_rows:
        return {e: 0.0 for e in emotions}
    return {
        e: sum(r[e] for r in matched_rows) / len(matched_rows)
        for e in emotions
    }


# Build fast lookup dict: word -> {emotion: intensity}
EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
intensity_lookup = {
    row['word']: {e: row[e] for e in EMOTIONS}
    for _, row in nrc_intensity_wide.iterrows()
}

# Score all unique TMDB keywords using phrase-level logic
keyword_scores = {
    kw: score_keyword_phrase(kw, intensity_lookup, EMOTIONS)
    for kw in unique_keywords
}

# Convert to DataFrame for inspection
kw_scores_df = pd.DataFrame.from_dict(keyword_scores, orient='index')
kw_scores_df.index.name = 'keyword'
kw_scores_df = kw_scores_df.reset_index()

covered = (kw_scores_df[EMOTIONS].sum(axis=1) > 0).sum()
total = len(kw_scores_df)
print(f'Keywords with at least one emotion signal: {covered:,} / {total:,} ({100*covered/total:.1f}%)')
print()
# Show phrase examples to verify lemmatization works
examples = ['coming of age', 'post apocalypse', 'serial killer', 'rescue mission', 'love story', 'outer space']
subset = kw_scores_df[kw_scores_df['keyword'].isin(examples)]
if not subset.empty:
    print(subset.set_index('keyword').to_string())
else:
    print('(none of the example keywords found in sample)')


Keywords with at least one emotion signal: 7,482 / 16,298 (45.9%)

                 anger  anticipation  disgust   fear    joy  sadness  surprise  trust
keyword                                                                              
coming of age      0.0         0.633      0.0  0.000  0.000      0.0     0.000  0.000
love story         0.0         0.000      0.0  0.000  0.828      0.0     0.000  0.758
outer space        0.0         0.000      0.0  0.000  0.000      0.0     0.000  0.000
post apocalypse    0.0         0.000      0.0  0.844  0.000      0.0     0.000  0.000
rescue mission     0.0         0.664      0.0  0.000  0.225      0.0     0.547  0.531
serial killer      0.0         0.438      0.0  0.000  0.000      0.0     0.000  0.000


## NRC VAD Lexicon (Valence–Arousal–Dominance)

[NRC VAD Lexicon](https://www.saifmohammad.com/WebPages/nrc-vad.html): 20,007 English words scored on three affective dimensions (0–1):
- **Valence** — positivity (0=negative, 1=positive)
- **Arousal** — energy/activation (0=calm, 1=excited)
- **Dominance** — control/power (0=submissive, 1=dominant)

File: `data/NRC-VAD-Lexicon/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt` (tab-separated, no header: word, valence, arousal, dominance).

In [18]:
from pathlib import Path

VAD_FILE = Path('data/NRC-VAD-Lexicon/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt')

nrc_vad = pd.read_csv(
    VAD_FILE, sep='\t', header=None,
    names=['word', 'valence', 'arousal', 'dominance']
)
print(f'NRC VAD: {len(nrc_vad):,} words')
nrc_vad.head(10)

NRC VAD: 19,971 words


,word,valence,arousal,dominance
0,aaaaaaah,0.479,0.606,0.291
1,aaaah,0.520,0.636,0.282
2,aardvark,0.427,0.490,0.437
3,aback,0.385,0.407,0.288
4,abacus,0.510,0.276,0.485
5,abalone,0.500,0.480,0.412
6,abandon,0.052,0.519,0.245
7,abandoned,0.046,0.481,0.130
8,abandonment,0.128,0.430,0.202
9,abashed,0.177,0.644,0.307


In [19]:
# Build VAD lookup dict (reuses _tokenize and _lemma_candidates from phrase-keyword-scoring)
VAD_DIMS = ['valence', 'arousal', 'dominance']

vad_lookup = {
    row['word']: {d: row[d] for d in VAD_DIMS}
    for _, row in nrc_vad.iterrows()
}


def score_keyword_vad(keyword: str, vad_lookup: dict, dims: list) -> dict:
    """
    Score a keyword phrase on VAD dimensions.
    Same phrase-tokenize + lemmatize strategy as score_keyword_phrase().
    Returns mean VAD scores across matched tokens; NaN if no match
    (VAD scores represent a scale midpoint at 0.5, so NaN is preferable to 0).
    """
    tokens = _tokenize(keyword)
    matched_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in vad_lookup:
                matched_rows.append(vad_lookup[lemma])
                break
    if not matched_rows:
        return {d: float('nan') for d in dims}
    return {
        d: sum(r[d] for r in matched_rows) / len(matched_rows)
        for d in dims
    }


# Score all unique keywords
kw_vad_scores = {
    kw: score_keyword_vad(kw, vad_lookup, VAD_DIMS)
    for kw in unique_keywords
}

kw_vad_df = pd.DataFrame.from_dict(kw_vad_scores, orient='index')
kw_vad_df.index.name = 'keyword'
kw_vad_df = kw_vad_df.reset_index()

covered_vad = kw_vad_df['valence'].notna().sum()
total = len(kw_vad_df)
print(f'Keywords with VAD scores: {covered_vad:,} / {total:,} ({100*covered_vad/total:.1f}%)')
print()
examples = ['coming of age', 'serial killer', 'love story', 'rescue mission', 'outer space', 'ghost']
subset = kw_vad_df[kw_vad_df['keyword'].isin(examples)]
if not subset.empty:
    print(subset.set_index('keyword').round(3).to_string())
else:
    print('(none of the example keywords found in sample)')

Keywords with VAD scores: 13,699 / 16,298 (84.1%)

                valence  arousal  dominance
keyword                                    
coming of age     0.607    0.322      0.499
ghost             0.265    0.755      0.343
love story        0.826    0.462      0.688
outer space       0.676    0.394      0.514
rescue mission    0.786    0.780      0.860
serial killer     0.255    0.660      0.508


## Movie-Level Sentiment: Valence Aggregation

Aggregate per-keyword VAD valence scores up to movie level by averaging across all keywords.
Valence > 0.5 = positive, < 0.5 = negative, ≈ 0.5 = neutral.

In [20]:
EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD_DIMS = ['valence', 'arousal', 'dominance']

def score_movie(keywords_str: str) -> dict:
    """Score a movie's keywords → mean emotion intensity + mean VAD scores."""
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]

    # Emotion intensity: mean of non-zero signal per emotion
    emotion_rows = [keyword_scores[kw] for kw in kws if kw in keyword_scores]
    emotion_means = {}
    if emotion_rows:
        for e in EMOTIONS:
            vals = [r[e] for r in emotion_rows if r[e] > 0]
            emotion_means[e] = sum(vals) / len(vals) if vals else 0.0

    # VAD: mean across matched keywords (NaN-safe)
    vad_rows = [kw_vad_scores[kw] for kw in kws if kw in kw_vad_scores]
    vad_means = {}
    for d in VAD_DIMS:
        vals = [r[d] for r in vad_rows if r[d] == r[d]]  # exclude NaN
        vad_means[d] = sum(vals) / len(vals) if vals else float('nan')

    return {**emotion_means, **vad_means}


# Score all movies
scores = df['keywords'].map(score_movie).apply(pd.Series)
movies = pd.concat([df[['id', 'title', 'keywords']], scores], axis=1)

# Sentiment from valence
movies['sentiment'] = movies['valence'].apply(
    lambda v: 'positive' if v > 0.5 else ('negative' if v < 0.5 else 'neutral')
    if v == v else 'unknown'
)

print(f'Movies scored: {movies["valence"].notna().sum():,} / {len(movies):,}')
print(f'\nSentiment distribution:')
print(movies['sentiment'].value_counts().to_string())
print()
print('Sample — most positive valence:')
display(movies.nlargest(5, 'valence')[['title', 'valence', 'arousal', 'dominance', 'sentiment']])
print('Sample — most negative valence:')
display(movies.nsmallest(5, 'valence')[['title', 'valence', 'arousal', 'dominance', 'sentiment']])

Movies scored: 9,489 / 10,000

Sentiment distribution:
sentiment
positive    5954
negative    3534
unknown      511
neutral        1

Sample — most positive:


,title,keywords,valence,sentiment
5811,Little Italy,love,1.0,positive
5827,A Perfect Plan,"france, love",1.0,positive
7641,Ma che bella sorpresa,love,1.0,positive
8069,Skin,"skinhead, love",1.0,positive
8627,A Short Film About Love,love,1.0,positive


Sample — most negative:


,title,keywords,valence,sentiment
9958,Narco Sub,criminal,0.021,negative
9979,Killshot,"hitman, criminal",0.021,negative
5901,Devotion,"korean war, aftercreditsstinger, duringcredits...",0.022,negative
6208,Return of the Hero,napoleonic wars,0.022,negative
7463,The Last Full Measure,vietnam war,0.022,negative


## Output: Write CSV + Upload to Kaggle

Writes `output/movie_vad_scores.csv` then pushes it to Kaggle as a new dataset version.
Set `NROWS = None` at the top of the notebook to score all ~930K movies.

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'movie_vad_scores.csv'

# Select and order output columns
out_cols = ['id', 'title', 'keywords', 'sentiment', 'valence', 'arousal', 'dominance'] + EMOTIONS
movies[out_cols].to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(movies):,} rows → {OUTPUT_CSV}')
print(f'Columns: {out_cols}')
movies[out_cols].head(5)

In [ ]:
import subprocess, json
from pathlib import Path

DATASET_SLUG = 'tmdb-movie-vad-emotion-scores'
OUTPUT_DIR = Path('output')
META_FILE = OUTPUT_DIR / 'dataset-metadata.json'

# Kaggle username from env
kaggle_username = os.environ.get('KAGGLE_USERNAME', '')
if not kaggle_username:
    # Try reading from kaggle.json
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if kaggle_json.exists():
        kaggle_username = json.loads(kaggle_json.read_text()).get('username', '')

if not kaggle_username:
    print('Set KAGGLE_USERNAME in .env to enable upload.')
else:
    meta = {
        'title': 'TMDB Movie VAD + Emotion Scores',
        'id': f'{kaggle_username}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
        'description': (
            '~930K TMDB movies scored with NRC VAD (valence/arousal/dominance) '
            'and NRC Emotion Intensity (anger, fear, joy, sadness, disgust, '
            'surprise, anticipation, trust) derived from movie keywords. '
            'Valence > 0.5 = positive sentiment.'
        ),
    }
    META_FILE.write_text(json.dumps(meta, indent=2))
    print(f'Metadata written: {META_FILE}')
    print(f'Dataset id: {meta["id"]}')
    print()
    print('To create (first time):')
    print(f'  kaggle datasets create -p {OUTPUT_DIR}')
    print('To update (subsequent runs):')
    print(f'  kaggle datasets version -p {OUTPUT_DIR} -m "Updated scores"')

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10000 non-null  int64  
 1   title                 10000 non-null  str    
 2   vote_average          10000 non-null  float64
 3   vote_count            10000 non-null  int64  
 4   status                10000 non-null  str    
 5   release_date          10000 non-null  str    
 6   revenue               10000 non-null  int64  
 7   runtime               10000 non-null  int64  
 8   adult                 10000 non-null  bool   
 9   backdrop_path         9988 non-null   str    
 10  budget                10000 non-null  int64  
 11  homepage              4096 non-null   str    
 12  imdb_id               9995 non-null   str    
 13  original_language     10000 non-null  str    
 14  original_title        10000 non-null  str    
 15  overview              9999 non-